In [0]:
from pyspark.sql import functions as F

input_file_path = "/Volumes/slowly_changing_dim/source/tutorial_volume/scd1/input_files/"
target_table = "slowly_changing_dim.bronze.scd1"
checkpoint_path = "/Volumes/slowly_changing_dim/source/tutorial_volume/scd1/checkpoints/"
schema_location  = "/Volumes/slowly_changing_dim/source/tutorial_volume/scd1/schema/"

Reset checkpoints, shcema and catalog tables if exists

In [0]:
# dbutils.fs.rm(checkpoint_path, True)
# dbutils.fs.rm(schema_location, True)
# spark.sql(f"DROP TABLE IF EXISTS {target_table}")

In [0]:
df = (
    spark.readStream
    .format('cloudFiles')
    .option('cloudFiles.schemaLocation', schema_location)
    .option('cloudFiles.format', 'csv')
    .option('header', True)
    .option('inferScehma', True)
    .load(input_file_path)
    .withColumn('read_timestamp', F.current_timestamp())
    .withColumn('file_name', F.lit(input_file_path))
)

In [0]:
query = (
    df.writeStream
    .format('delta')
    .outputMode('append')
    .option('checkpointLocation', checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()

print(f"Stream is terminated and data is successfully loaded to {target_table}")

In [0]:
# print(query.lastProgress)

In [0]:
%sql
SELECT COUNT(*) FROM slowly_changing_dim.bronze.scd1

In [0]:
%sql
SELECT * FROM slowly_changing_dim.bronze.scd1 LIMIT 10